In [ ]:
%load_ext autoreload
%autoreload 2
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from compactreasoningmodels.utils.grid import pad_clues

In [ ]:
if "original_dir" not in globals():
    original_dir = os.getcwd()

os.chdir(os.path.join(original_dir, ".."))
df = pd.read_json("data/traces/large_dataset.jsonl", lines=True)
solvers = list(df["traces"][0].keys())
lookup = {solver: torch.tensor(i, dtype=torch.long) for i, solver in enumerate(solvers)}

In [ ]:
solver_X = {}
solver_y = {}
for solver_name in solvers:
    X, y = [], []
    for _, row in df.iterrows():
        X.append(pad_clues(row["clues"]).flatten())
        y.append(torch.tensor(row["traces"][solver_name]["1.0"][0]["steps"]).flatten())
    solver_X[solver_name] = torch.stack(X)
    solver_y[solver_name] = torch.stack(y)

In [ ]:
datasets = {
    solver_name: TensorDataset(solver_X[solver_name], solver_y[solver_name])
    for solver_name in solvers
}
train_datasets = {
    solver_name: TensorDataset(
        solver_X[solver_name][: int(0.8 * len(solver_X[solver_name]))],
        solver_y[solver_name][: int(0.8 * len(solver_y[solver_name]))],
    )
    for solver_name in solvers
}
test_datasets = {
    solver_name: TensorDataset(
        solver_X[solver_name][int(0.8 * len(solver_X[solver_name])) :],
        solver_y[solver_name][int(0.8 * len(solver_y[solver_name])) :],
    )
    for solver_name in solvers
}
train_loaders = {
    solver_name: DataLoader(train_datasets[solver_name], batch_size=128, shuffle=True)
    for solver_name in solvers
}
test_loaders = {
    solver_name: DataLoader(test_datasets[solver_name], batch_size=128, shuffle=False)
    for solver_name in solvers
}

In [ ]:
class MLPModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out


mlp_models = {
    solver_name: MLPModel(
        input_size=solver_X[solver_name].shape[1],
        hidden_size=128,
        output_size=solver_y[solver_name].shape[1],
    )
    for solver_name in solvers
}

In [ ]:
class LinearModel(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.fc(x)


linear_models = {
    solver_name: LinearModel(
        input_size=solver_X[solver_name].shape[1], output_size=solver_y[solver_name].shape[1]
    )
    for solver_name in solvers
}

In [ ]:
class StackedModel(nn.Module):
    def __init__(
        self, vocab_size, seq_len, hidden_clues_size, hidden_size, output_size, num_steps=15
    ):
        super().__init__()
        self.num_steps = num_steps
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.seq_len = seq_len

        assert output_size % num_steps == 0, (
            f"output_size ({output_size}) must be divisible by num_steps ({num_steps})"
        )
        self.grid_size = output_size // num_steps

        self.embedding = nn.Embedding(vocab_size, hidden_clues_size)
        clue_dim = seq_len * hidden_clues_size
        self.unembedding = nn.Linear(hidden_size, self.grid_size)
        self.steps = nn.ModuleList(
            [nn.Linear(hidden_size + clue_dim, hidden_size) for _ in range(self.num_steps)]
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        # x: (batch, seq_len) integer clue indices
        x = x.long()
        batch_size = x.shape[0]
        clue_embedding = self.embedding(x).flatten(
            start_dim=1
        )  # (batch, seq_len * hidden_clues_size)

        hidden_state = torch.full(
            (batch_size, self.hidden_size),
            0.5,
            dtype=torch.float32,
            device=x.device,
        )

        out = []
        for step in self.steps:
            hidden_state = step(torch.cat((hidden_state, clue_embedding), dim=1))
            hidden_state = self.relu(hidden_state)
            out.append(self.unembedding(hidden_state))

        return torch.cat(out, dim=1)  # (batch, num_steps * grid_size)


stacked_models = {
    solver_name: StackedModel(
        vocab_size=int(solver_X[solver_name].max()) + 1,
        seq_len=solver_X[solver_name].shape[1],
        hidden_clues_size=64,
        hidden_size=128,
        output_size=solver_y[solver_name].shape[1],
    )
    for solver_name in solvers
}

In [ ]:
split_y = {solver_name: solver_y[solver_name].view(-1, 15, 25) for solver_name in solvers}

# Create datasets for each of the 15 slices
split_dataset = {
    solver_name: [
        TensorDataset(solver_X[solver_name], split_y[solver_name][:, i, :]) for i in range(15)
    ]
    for solver_name in solvers
}

# Train/test split (FIXED)
split_ratio = 0.8
split_train_datasets = {
    solver_name: [
        TensorDataset(
            solver_X[solver_name][: int(split_ratio * len(solver_X[solver_name]))],
            split_y[solver_name][: int(split_ratio * len(split_y[solver_name])), i, :],
        )
        for i in range(15)
    ]
    for solver_name in solvers
}

split_test_datasets = {
    solver_name: [
        TensorDataset(
            solver_X[solver_name][int(split_ratio * len(solver_X[solver_name])) :],
            split_y[solver_name][
                int(split_ratio * len(split_y[solver_name])) :, i, :
            ],  # FIXED: added colon
        )
        for i in range(15)
    ]
    for solver_name in solvers
}

# DataLoaders
split_train_loaders = {
    solver_name: [
        DataLoader(split_train_datasets[solver_name][i], batch_size=128, shuffle=True)
        for i in range(15)
    ]
    for solver_name in solvers
}

split_test_loaders = {
    solver_name: [
        DataLoader(split_test_datasets[solver_name][i], batch_size=128, shuffle=False)
        for i in range(15)
    ]
    for solver_name in solvers
}

# Models
split_linear_models = {
    solver_name: [
        LinearModel(input_size=solver_X[solver_name].shape[1], output_size=25) for i in range(15)
    ]
    for solver_name in solvers
}

In [ ]:
def train_model(model, train_loader, criterion, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()


def evaluate_model(model, test_loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()
    return total_loss / len(test_loader)

In [ ]:
for solver_name in solvers:
    for i in range(15):
        model = split_linear_models[solver_name][i]
        train_loader = split_train_loaders[solver_name][i]
        test_loader = split_test_loaders[solver_name][i]
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

        train_model(model, train_loader, criterion, optimizer, num_epochs=50)
        test_loss = evaluate_model(model, test_loader, criterion)
        print(f"Solver: {solver_name}, Step: {i}, Test Loss: {test_loss}")

In [ ]:
for solver_name in solvers:
    print(f"Training Stacked model for solver: {solver_name}")
    model = stacked_models[solver_name]
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    train_model(model, train_loaders[solver_name], criterion, optimizer, num_epochs=10)
    test_loss = evaluate_model(model, test_loaders[solver_name], criterion)
    print(f"Test loss for solver {solver_name}: {test_loss}")

In [ ]:
import torch
import torch.nn.functional as F


def linear_cka(X, Y):
    """
    X: (N, d1), Y: (N, d2) — activations from two models on the same N inputs.
    Standard linear CKA: ||Y^T X||_F^2 / (||X^T X||_F * ||Y^T Y||_F)
    """
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)

    xty_f2 = torch.norm(X.T @ Y, p="fro") ** 2
    xtx_f = torch.norm(X.T @ X, p="fro")
    yty_f = torch.norm(Y.T @ Y, p="fro")

    return (xty_f2 / (xtx_f * yty_f)).item()


def debiased_cka(X, Y):
    """
    Unbiased linear CKA (Kornblith et al. 2019, based on Szekely & Rizzo's
    unbiased HSIC estimator). Reduces small-N inflation seen in naive CKA.
    """

    def unbiased_hsic(K, L):
        n = K.shape[0]
        K = K.clone()
        L = L.clone()
        K.fill_diagonal_(0)
        L.fill_diagonal_(0)
        ones = torch.ones(n, device=K.device)
        trace_term = torch.trace(K @ L)
        sum_term = (ones @ K @ ones) * (ones @ L @ ones) / ((n - 1) * (n - 2))
        mid_term = ones @ K @ L @ ones * 2 / (n - 2)
        return (trace_term + sum_term - mid_term) / (n * (n - 3))

    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    K = X @ X.T
    L = Y @ Y.T

    hsic_xy = unbiased_hsic(K, L)
    hsic_xx = unbiased_hsic(K, K)
    hsic_yy = unbiased_hsic(L, L)

    return (hsic_xy / torch.sqrt(hsic_xx * hsic_yy)).item()


@torch.no_grad()
def stacked_model_activations(model, loader, device):
    """
    Run a StackedModel over a loader and collect post-ReLU hidden states
    at every step. Mirrors StackedModel.forward's internal loop instead of
    duplicating a fixed fc1/relu assumption.

    Returns: tensor of shape (num_steps, N, hidden_size)
    """
    model.eval()
    model.to(device)
    per_step_acts = [[] for _ in range(model.num_steps)]

    for x, _ in loader:
        x = x.to(device).long()
        batch_size = x.shape[0]
        clue_embedding = model.embedding(x).flatten(start_dim=1)

        hidden_state = torch.full(
            (batch_size, model.hidden_size),
            0.5,
            dtype=torch.float32,
            device=device,
        )

        for step_idx, step in enumerate(model.steps):
            hidden_state = step(torch.cat((hidden_state, clue_embedding), dim=1))
            hidden_state = model.relu(hidden_state)
            per_step_acts[step_idx].append(hidden_state.cpu())

    return torch.stack([torch.cat(acts, dim=0) for acts in per_step_acts], dim=0)


def cka_similarity_matrix(models, loader, device):
    """
    models: dict[name -> StackedModel], all evaluated on the same loader/inputs.
    Returns:
        sim: tensor (num_steps, n_models, n_models)
        names: list of model names, matching sim's index order
    """
    names = list(models.keys())
    # (num_steps, N, hidden_size) per model
    all_acts = {name: stacked_model_activations(m, loader, device) for name, m in models.items()}

    num_steps = next(iter(all_acts.values())).shape[0]
    n = len(names)
    sim = torch.zeros(num_steps, n, n)

    for s in range(num_steps):
        for i, ni in enumerate(names):
            for j, nj in enumerate(names):
                if j < i:
                    sim[s, i, j] = sim[s, j, i]  # symmetric, skip recompute
                else:
                    sim[s, i, j] = debiased_cka(all_acts[ni][s], all_acts[nj][s])

    return sim, names

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# probe_loader must yield the SAME (x, y) batches regardless of which model
# consumes them — same puzzle instances, same batch order, no shuffling.
probe_loader = torch.utils.data.DataLoader(test_datasets["mac"], batch_size=64, shuffle=False)

sim, names = cka_similarity_matrix(stacked_models, probe_loader, device)

# sim: (num_steps, n, n). Print the final step's similarity matrix:
print("Model order:", names)
print("CKA at final step:\n", sim[0])

# Similarity trajectory between two specific solvers across all 15 steps:
i, j = names.index("mac"), names.index("model_solver")
print("Per-step CKA, solver_a vs solver_b:", sim[:, i, j])

In [ ]:
import matplotlib.pyplot as plt


def plot_cka_steps(sim, names, vmin=0.0, vmax=1.0, cmap="viridis", figsize_per_cell=1.8):
    """
    sim: (num_steps, n_models, n_models) tensor/array
    names: list of model names, length n_models
    vmin/vmax: shared color scale across all subplots so steps are visually comparable
    """
    sim = sim.detach().cpu().numpy() if hasattr(sim, "detach") else np.asarray(sim)
    num_steps, n, _ = sim.shape

    ncols = 5
    nrows = int(np.ceil(num_steps / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * figsize_per_cell, nrows * figsize_per_cell),
        squeeze=False,
    )

    im = None
    for s in range(num_steps):
        ax = axes[s // ncols, s % ncols]
        im = ax.imshow(sim[s], vmin=vmin, vmax=vmax, cmap=cmap)
        ax.set_title(f"step {s}", fontsize=10)
        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        # ax.set_xticklabels(names, rotation=90, fontsize=6)
        # ax.set_yticklabels(names, fontsize=6)

    # hide unused subplots if num_steps doesn't fill the grid
    for s in range(num_steps, nrows * ncols):
        axes[s // ncols, s % ncols].axis("off")

    fig.colorbar(im, ax=axes, shrink=0.6, label="CKA")
    fig.suptitle("Pairwise CKA similarity across steps", fontsize=14)
    return fig


fig = plot_cka_steps(sim, names)
plt.show()

In [ ]:
def minmax_normalize_per_step(sim):
    s = sim.clone()
    n = s.shape[-1]
    mask = ~torch.eye(n, dtype=torch.bool, device=s.device)
    for step in range(s.shape[0]):
        vals = s[step][mask]
        vmin, vmax = vals.min(), vals.max()
        s[step] = (s[step] - vmin) / (vmax - vmin + 1e-8)
        s[step].fill_diagonal_(1.0)  # keep self-similarity at 1 after rescale
    return s


normalized_sim = minmax_normalize_per_step(sim)
fig = plot_cka_steps(normalized_sim, names, vmin=0.0, vmax=1.0)
plt.show()

In [ ]:
baseline_a = StackedModel(
    vocab_size=int(solver_X[solver_name].max()) + 1,
    seq_len=solver_X[solver_name].shape[1],
    hidden_clues_size=64,
    hidden_size=128,
    output_size=solver_y[solver_name].shape[1],
)
baseline_b = StackedModel(
    vocab_size=int(solver_X[solver_name].max()) + 1,
    seq_len=solver_X[solver_name].shape[1],
    hidden_clues_size=64,
    hidden_size=128,
    output_size=solver_y[solver_name].shape[1],
)
sim_baseline, _ = cka_similarity_matrix(
    {"rand_a": baseline_a, "rand_b": baseline_b}, probe_loader, device
)
print("Random-init CKA baseline:", sim_baseline[-1])

In [ ]:


@torch.no_grad()
def output_similarity(models, loader, device):
    names = list(models.keys())
    all_outputs = {name: [] for name in names}

    for x, _ in loader:
        x = x.to(device)
        for name, model in models.items():
            model.eval()
            all_outputs[name].append(model(x).cpu())

    for name in names:
        all_outputs[name] = torch.cat(all_outputs[name], dim=0)  # [N, output_size]

    n = len(names)
    sim = torch.zeros(n, n)
    for i, ni in enumerate(names):
        for j, nj in enumerate(names):
            a, b = all_outputs[ni], all_outputs[nj]
            # cosine similarity per-sample, averaged
            sim[i, j] = F.cosine_similarity(a, b, dim=1).mean()
    return sim, names


output_similarity(models, test_loaders["genetic_algorithm_det"], device="cpu")

In [ ]:
import matplotlib.pyplot as plt

from compactreasoningmodels.utils.display import display_grid

solver_name = "mac"
X1, y1 = next(iter(test_loaders[solver_name]))
X1, y1 = X1.to("cpu")[0], y1.to("cpu")[0]
X1 = X1.reshape(2, 5, 3)
y1 = y1.reshape(15, 5, 5)

fig, axs = plt.subplots(3, 5, figsize=(15, 6))
for i in range(3):
    for j in range(5):
        display_grid(ax=axs[i, j], grid=y1[i * 3 + j], clues=X1)

# predicted_y1 = models[solver_name](X1.flatten().unsqueeze(0)).reshape(15, 5, 5)
predicted_y1 = [
    split_linear_models[solver_name][i](X1.flatten().unsqueeze(0)).reshape(5, 5) for i in range(15)
]
fig, axs = plt.subplots(3, 5, figsize=(15, 6))
for i in range(3):
    for j in range(5):
        display_grid(ax=axs[i, j], grid=predicted_y1[i * 3 + j], clues=X1)